<a href="https://colab.research.google.com/github/Hackathon-D-Group-2026/Hackathon_D_Group_2026/blob/main/sai%20krishna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
CAT_FEATURES = [
    "age_group", "education", "race", "sex", "income_poverty",
    "marital_status", "rent_or_own", "employment_status", "census_msa", "employment_sector"
]
BEH_FEATURES = [c for c in train_data.columns if c.startswith("behavioral_")]


def transform_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.drop(columns=[ID_FIELD, TARGET], errors="ignore").copy()
    out["hygiene_score"] = out[BEH_FEATURES].sum(axis=1, min_count=1)
    doc = out["doctor_recc_seasonal"].fillna(0)
    eff = out["opinion_seas_vacc_effective"].fillna(3)
    rsk = out["opinion_seas_risk"].fillna(3)
    sck = out["opinion_seas_sick_from_vacc"].fillna(3)
    out["doc_x_effective"] = doc * eff
    out["doc_x_risk"] = doc * rsk
    out["attitude_net_score"] = (eff + rsk) - sck
    ru_tmp, rv_tmp = estimate_flu_hosp_risks(df)
    out["expected_hosp_saving"] = (ru_tmp - rv_tmp) * H_COST
    out["na_row_count"] = out.isna().sum(axis=1)
    for c in CAT_FEATURES:
        out[c] = out[c].astype("object")
    return out


X_tr = transform_domain_features(train_data)
X_te = transform_domain_features(test_data)
NUM_FEATURES = [c for c in X_tr.columns if c not in CAT_FEATURES]
CAT_POS = list(range(len(CAT_FEATURES)))

Pipe_Linear_Flag = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=True)),
        ("sc", RobustScaler()),
    ]), NUM_FEATURES),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="constant", fill_value="Unreported")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=15)),
    ]), CAT_FEATURES),
])

Pipe_Linear_Simple = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=False)),
        ("sc", StandardScaler()),
    ]), NUM_FEATURES),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=15)),
    ]), CAT_FEATURES),
])

Pipe_Tree_Native = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1), CAT_FEATURES),
    ("num", "passthrough", NUM_FEATURES),
])

Pipe_Tree_Sentinel = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1), CAT_FEATURES),
    ("num", SimpleImputer(strategy="constant", fill_value=-1), NUM_FEATURES),
])

print(f"Engineered Feature Matrix: {X_tr.shape[1]} features ({len(NUM_FEATURES)} numeric + {len(CAT_FEATURES)} categorical)")

Engineered Feature Matrix: 35 features (25 numeric + 10 categorical)


In [ ]:
CV_20 = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=RNG_SEED)

cv_logs = {}
oof_predictions = {}


def run_cv_evaluation(model_label: str, pipeline_obj, X_in=X_tr, y_in=y_obs, cv_splitter=CV_20, show_log=True):
    """Run 20-fold repeated stratified CV and compute fold-wise town expenditure per resident."""
    fold_records = []
    q_accum = np.zeros(len(y_in), dtype=float)
    q_counts = np.zeros(len(y_in), dtype=float)

    for f_id, (idx_tr, idx_va) in enumerate(cv_splitter.split(X_in, y_in)):
        fitted_m = clone(pipeline_obj).fit(X_in.iloc[idx_tr], y_in[idx_tr])
        q_hat = fitted_m.predict_proba(X_in.iloc[idx_va])[:, 1]
        ru_v, rv_v, y_v = RU_TR[idx_va], RV_TR[idx_va], y_in[idx_va]

        s_personal = allocate_vaccine(q_hat, ru_v, rv_v)
        s_half = (q_hat >= 0.50).astype(int)

        fold_records.append({
            "fold": f_id,
            "bill_per_res": municipal_flu_bill(y_v, s_personal, ru_v, rv_v) / len(idx_va),
            "bill_at_half": municipal_flu_bill(y_v, s_half, ru_v, rv_v) / len(idx_va),
            "logloss": log_loss(y_v, q_hat),
            "brier": brier_score_loss(y_v, q_hat),
            "auc": roc_auc_score(y_v, q_hat),
            "pct_sent": s_personal.mean() * 100.0,
        })
        q_accum[idx_va] += q_hat
        q_counts[idx_va] += 1

    df_out = pd.DataFrame(fold_records)
    cv_logs[model_label] = df_out
    oof_predictions[model_label] = q_accum / np.maximum(q_counts, 1)
    if show_log:
        print(
            f"{model_label:26s} | Bill/Res: ${df_out.bill_per_res.mean():7.2f} ± {df_out.bill_per_res.std():5.2f} "
            f"| Bill@0.5: ${df_out.bill_at_half.mean():7.2f} | LogLoss: {df_out.logloss.mean():.4f} | AUC: {df_out.auc.mean():.4f}"
        )
    return df_out

In [ ]:
X_raw_base = train_data.drop(columns=[ID_FIELD, TARGET]).copy()
for c in CAT_FEATURES:
    X_raw_base[c] = X_raw_base[c].astype("object")
RAW_NUM = [c for c in X_raw_base.columns if c not in CAT_FEATURES]

Pipe_Raw_Simple = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), RAW_NUM),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), CAT_FEATURES),
])

ablation_models = {
    "1. Raw Features + Naive Imputation": (Pipeline([("p", Pipe_Raw_Simple), ("m", LogisticRegression(C=0.2, max_iter=3000, random_state=RNG_SEED))]), X_raw_base),
    "2. Engineered Features + Naive Imputation": (Pipeline([("p", Pipe_Linear_Simple), ("m", LogisticRegression(C=0.2, max_iter=3000, random_state=RNG_SEED))]), X_tr),
    "3. Engineered + Missingness Flags (Pipe_Linear_Flag)": (Pipeline([("p", Pipe_Linear_Flag), ("m", LogisticRegression(C=0.2, max_iter=3000, random_state=RNG_SEED))]), X_tr),
}

ab_rows = []
for ab_name, (ab_pipe, ab_X) in ablation_models.items():
    df_ab = run_cv_evaluation(ab_name, ab_pipe, X_in=ab_X, show_log=False)
    ab_rows.append({
        "Pre-processing Configuration": ab_name,
        "20-Fold Mean Bill / Resident ($)": df_ab.bill_per_res.mean(),
        "Fold SD ($)": df_ab.bill_per_res.std(),
        "Log Loss": df_ab.logloss.mean(),
        "Brier Score": df_ab.brier.mean(),
        "ROC-AUC": df_ab.auc.mean(),
    })
    cv_logs.pop(ab_name, None)
    oof_predictions.pop(ab_name, None)

display(pd.DataFrame(ab_rows).set_index("Pre-processing Configuration").style.format({
    "20-Fold Mean Bill / Resident ($)": "${:,.2f}",
    "Fold SD ($)": "${:,.2f}",
    "Log Loss": "{:.4f}",
    "Brier Score": "{:.4f}",
    "ROC-AUC": "{:.4f}",
}))

,20-Fold Mean Bill / Resident ($),Fold SD ($),Log Loss,Brier Score,ROC-AUC
Pre-processing Configuration,,,,,
1. Raw Features + Naive Imputation,$514.32,$15.39,0.4870,0.1580,0.8475
2. Engineered Features + Naive Imputation,$514.27,$15.46,0.4875,0.1582,0.8471
3. Engineered + Missingness Flags (Pipe_Linear_Flag),$513.00,$15.39,0.4833,0.1566,0.8501
